# Bài 2: cài ptLasso, glmtrans trên Kaggle R (PD-04)

Notebook chỉ clone đúng commit đã push rồi gọi `scripts/r/check_r_baselines.R`: cài hai gói, chạy ví dụ binomial trong manual CRAN và ghi `sessionInfo()`.
Đây là kiểm tra môi trường trên dữ liệu mô phỏng của hai gói, chưa chạy baseline trên dữ liệu đề tài và chưa ép đúng k (M6).

Mỗi phiên:
1. Language: **R**. Accelerator: None. Bật **Internet** (cần để clone và tải gói từ CRAN).
2. Không cần gắn Input.
3. Sửa `REPO_COMMIT` ở ô cấu hình thành commit đã push.
4. Save Version → Save & Run All. Kết quả: `/kaggle/working/r_baselines_results.zip`.

Kết quả lỗi vẫn có giá trị: `r_baseline_check.csv` ghi lỗi cụ thể của từng bước để giảng viên quyết phương án thay thế.

In [ ]:
REPO_URL <- 'https://github.com/datachain-uit/CN-23520899-LuanNT.git'
REPO_BRANCH <- 'rq2-mvp'
REPO_COMMIT <- ''                    # bắt buộc: commit đã push, vd. '9a15860'

REPO_DIR <- '/kaggle/working/CN-23520899-LuanNT'
OUT_DIR <- '/kaggle/working/r_baselines'
LIB_DIR <- '/tmp/r_baselines_lib'    # ngoài /kaggle/working để gói đã cài không lọt vào output
R.version.string

In [ ]:
# Gate: có mạng; clone đúng commit đã push.
if (!nzchar(REPO_COMMIT)) stop('Điền REPO_COMMIT ở ô cấu hình')
online <- tryCatch({ readLines(url('https://cloud.r-project.org/'), n = 1L, warn = FALSE); TRUE }, error = function(e) FALSE)
if (!online) stop('Không tải được CRAN: bật Internet trong Settings của notebook')
unlink(REPO_DIR, recursive = TRUE)
if (system2('git', c('clone', '--quiet', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR)) != 0L) stop('git clone lỗi')
if (system2('git', c('-C', REPO_DIR, 'checkout', '--quiet', REPO_COMMIT)) != 0L) stop('Không có commit này trên repo')
commit_hash <- system2('git', c('-C', REPO_DIR, 'rev-parse', 'HEAD'), stdout = TRUE)
pushed <- system2('git', c('-C', REPO_DIR, 'branch', '-r', '--contains', 'HEAD'), stdout = TRUE)
if (length(pushed) == 0L) stop('Commit chưa có trên remote: kết quả sẽ không truy vết được')
cat('COMMIT', commit_hash, '\n')

In [ ]:
# Chạy script; không dừng ở đây để kết quả luôn được lưu ở ô sau. Cài lần đầu có thể mất vài phút vì biên dịch gói phụ thuộc.
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
t0 <- Sys.time()
script_status <- system2('Rscript', file.path(REPO_DIR, 'scripts/r/check_r_baselines.R'),
                         stdout = file.path(OUT_DIR, 'run_stdout.log'), stderr = file.path(OUT_DIR, 'run_stderr.log'),
                         env = c(paste0('R_BASELINE_OUT=', OUT_DIR), paste0('R_BASELINE_LIB=', LIB_DIR), paste0('REPO_COMMIT_HASH=', commit_hash)))
cat('SCRIPT EXIT', script_status, 'sau', round(as.numeric(difftime(Sys.time(), t0, units = 'mins')), 1), 'phút\n')
cat(tail(readLines(file.path(OUT_DIR, 'run_stdout.log'), warn = FALSE), 40), sep = '\n')

In [ ]:
# Lưu kết quả rồi mới báo lỗi, để lỗi cài đặt vẫn tải về được.
archive <- '/kaggle/working/r_baselines_results.zip'
unlink(archive)
old <- setwd(dirname(OUT_DIR))
utils::zip(archive, basename(OUT_DIR), flags = '-r9Xq')
setwd(old)
cat('ARCHIVE', archive, file.size(archive), 'byte\n')
cat(readLines(file.path(OUT_DIR, 'sessionInfo.txt'), warn = FALSE), sep = '\n')
if (script_status != 0L) stop('Có bước lỗi: xem r_baseline_check.csv và run_stderr.log trong zip')
cat('R_BASELINES_OK\n')